# 05 — Propensity Model Experiments

## Objective
Iterate on the propensity baseline from notebook 04 without duplicating the whole pipeline.

**Important evaluation rule**
- Train on `train`.
- Compare/tune experiments on `validation`.
- Use `test` only once for the final selected candidate.


## 1. Install dependencies


In [ ]:
!pip -q install pyarrow scikit-learn joblib


## 2. Mount Drive and import reusable project code


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

PROJECT_DIR = Path("/content/drive/MyDrive/03 Projects/Factored Hackathon")
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from propensity.config import (
    RANDOM_STATE,
    TARGET,
    DATASET_PATH,
    ARTIFACT_DIR,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    MODEL_FEATURES,
)
from propensity.data import (
    load_dataset,
    validate_columns,
    temporal_split,
    make_xy,
)
from propensity.modeling import LogisticConfig, train_logistic
from propensity.evaluation import evaluate_model
from propensity.persistence import (
    save_model_and_metadata,
    build_scored_test,
    save_scored_test,
)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Load dataset and reproduce the same time-aware split


In [ ]:
df = load_dataset(DATASET_PATH)
validate_columns(df, MODEL_FEATURES, TARGET)

split = temporal_split(df)

X_train, y_train = make_xy(split.train_df, MODEL_FEATURES, TARGET)
X_val, y_val = make_xy(split.validation_df, MODEL_FEATURES, TARGET)
X_test, y_test = make_xy(split.test_df, MODEL_FEATURES, TARGET)

for name, frame in [
    ("Train", split.train_df),
    ("Validation", split.validation_df),
    ("Test", split.test_df),
]:
    print(
        f"{name:10s} rows={len(frame):,} "
        f"conversion_rate={frame[TARGET].mean():.4%}"
    )


Train      rows=849,821 conversion_rate=0.7324%
Validation rows=182,104 conversion_rate=0.7457%
Test       rows=182,105 conversion_rate=0.6908%


## 4. Define experiments

All four configurations are trained in one run. Each experiment gets its own artifact folder so results are never overwritten.

Model selection is based on validation only; test remains a one-time final evaluation.


In [ ]:
experiments = [
    LogisticConfig(
        name="baseline_reproduction",
        C=1.0,
        class_weight="balanced",
        max_iter=500,
        solver="saga",
        min_frequency=10,
        random_state=RANDOM_STATE,
    ),
    LogisticConfig(
        name="stronger_regularization",
        C=0.1,
        class_weight="balanced",
        max_iter=500,
        solver="saga",
        min_frequency=10,
        random_state=RANDOM_STATE,
    ),
    LogisticConfig(
        name="weaker_regularization",
        C=10.0,
        class_weight="balanced",
        max_iter=500,
        solver="saga",
        min_frequency=10,
        random_state=RANDOM_STATE,
    ),
    LogisticConfig(
        name="rarer_categories_grouped",
        C=1.0,
        class_weight="balanced",
        max_iter=500,
        solver="saga",
        min_frequency=50,
        random_state=RANDOM_STATE,
    ),
]

# One unique folder per notebook execution: reruns never overwrite older artifacts.
RUN_ID = pd.Timestamp.now(tz="America/Lima").strftime("%Y%m%d_%H%M%S")
RUN_DIR = Path(ARTIFACT_DIR) / "experiments" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Experiment run:", RUN_ID)
print("Artifacts will be saved under:", RUN_DIR)


Experiment run: 20261004_210515
Artifacts will be saved under: /content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/experiments/20261004_210515


## 5. Train all experiments and compare train + validation

Train metrics diagnose underfitting/overfitting. Validation metrics are used for model selection. Test is not touched here.


In [ ]:
trained_models = {}
train_results = {}
validation_results = {}
rows = []

for config in experiments:
    print("\n" + "=" * 80)
    print("Experiment:", config.name)

    model = train_logistic(
        X_train,
        y_train,
        NUMERIC_FEATURES,
        CATEGORICAL_FEATURES,
        config,
    )

    train_result = evaluate_model(
        model,
        X_train,
        y_train,
        name=f"{config.name} — TRAIN",
    )
    val_result = evaluate_model(
        model,
        X_val,
        y_val,
        name=f"{config.name} — VALIDATION",
    )

    trained_models[config.name] = model
    train_results[config.name] = train_result
    validation_results[config.name] = val_result

    # Persist each experiment independently inside this run.
    experiment_dir = RUN_DIR / config.name
    experiment_dir.mkdir(parents=True, exist_ok=True)

    experiment_metadata = {
        "run_id": RUN_ID,
        "target": TARGET,
        "numeric_features": NUMERIC_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
        "model_features": MODEL_FEATURES,
        "train_cutoff": str(split.train_cutoff),
        "validation_cutoff": str(split.validation_cutoff),
        "experiment": config.as_dict(),
        "train_metrics": {
            "pr_auc": train_result["pr_auc"],
            "roc_auc": train_result["roc_auc"],
            "base_conversion_rate": train_result["base_conversion_rate"],
            "best_f1": train_result["best_f1"],
            "best_f1_threshold": train_result["best_f1_threshold"],
        },
        "validation_metrics": {
            "pr_auc": val_result["pr_auc"],
            "roc_auc": val_result["roc_auc"],
            "base_conversion_rate": val_result["base_conversion_rate"],
            "best_f1": val_result["best_f1"],
            "best_f1_threshold": val_result["best_f1_threshold"],
            "precision_at_best_f1": val_result["precision_at_best_f1"],
            "recall_at_best_f1": val_result["recall_at_best_f1"],
        },
    }

    save_model_and_metadata(
        model,
        experiment_dir,
        experiment_metadata,
        model_filename="model.joblib",
        metadata_filename="metadata.json",
    )
    train_result["deciles"].to_csv(experiment_dir / "train_deciles.csv")
    val_result["deciles"].to_csv(experiment_dir / "validation_deciles.csv")

    rows.append({
        "experiment": config.name,
        "C": config.C,
        "class_weight": config.class_weight,
        "max_iter": config.max_iter,
        "min_frequency": config.min_frequency,
        "train_pr_auc": train_result["pr_auc"],
        "train_roc_auc": train_result["roc_auc"],
        "val_pr_auc": val_result["pr_auc"],
        "val_roc_auc": val_result["roc_auc"],
        "val_best_f1": val_result["best_f1"],
        "val_threshold": val_result["best_f1_threshold"],
    })

comparison = (
    pd.DataFrame(rows)
    .sort_values(["val_pr_auc", "val_roc_auc"], ascending=False)
    .reset_index(drop=True)
)

comparison_path = RUN_DIR / "comparison.csv"
comparison.to_csv(comparison_path, index=False)

print("\nComparison saved to:", comparison_path)
display(comparison)



Experiment: baseline_reproduction


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


baseline_reproduction — TRAIN
  PR-AUC / Average Precision: 0.007534
  ROC-AUC:                    0.508213
  Base conversion rate:       0.007324
baseline_reproduction — VALIDATION
  PR-AUC / Average Precision: 0.007574
  ROC-AUC:                    0.494867
  Base conversion rate:       0.007457

Experiment: stronger_regularization


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


stronger_regularization — TRAIN
  PR-AUC / Average Precision: 0.007268
  ROC-AUC:                    0.496720
  Base conversion rate:       0.007324
stronger_regularization — VALIDATION
  PR-AUC / Average Precision: 0.007517
  ROC-AUC:                    0.500596
  Base conversion rate:       0.007457

Experiment: weaker_regularization


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


weaker_regularization — TRAIN
  PR-AUC / Average Precision: 0.007064
  ROC-AUC:                    0.487914
  Base conversion rate:       0.007324
weaker_regularization — VALIDATION
  PR-AUC / Average Precision: 0.007387
  ROC-AUC:                    0.496032
  Base conversion rate:       0.007457

Experiment: rarer_categories_grouped


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


rarer_categories_grouped — TRAIN
  PR-AUC / Average Precision: 0.007534
  ROC-AUC:                    0.508213
  Base conversion rate:       0.007324
rarer_categories_grouped — VALIDATION
  PR-AUC / Average Precision: 0.007574
  ROC-AUC:                    0.494867
  Base conversion rate:       0.007457

Comparison saved to: /content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/experiments/20261004_210515/comparison.csv


,experiment,C,class_weight,max_iter,min_frequency,train_pr_auc,train_roc_auc,val_pr_auc,val_roc_auc,val_best_f1,val_threshold
0,baseline_reproduction,1.0,balanced,500,10,0.007534,0.508213,0.007574,0.494867,0.015433,0.998576
1,rarer_categories_grouped,1.0,balanced,500,50,0.007534,0.508213,0.007574,0.494867,0.015433,0.998576
2,stronger_regularization,0.1,balanced,500,10,0.007268,0.496720,0.007517,0.500596,0.015229,0.915895
3,weaker_regularization,10.0,balanced,500,10,0.007064,0.487914,0.007387,0.496032,0.015181,0.306660


## 6. Inspect the validation-selected candidate

The comparison table contains all four experiments. The candidate below is selected strictly from validation metrics.


In [ ]:
selected_name = comparison.loc[0, "experiment"]
selected_model = trained_models[selected_name]
selected_train = train_results[selected_name]
selected_validation = validation_results[selected_name]

print("Selected from validation:", selected_name)
print("Train ROC-AUC:", selected_train["roc_auc"])
print("Validation ROC-AUC:", selected_validation["roc_auc"])
print(
    "Best-F1 diagnostic threshold:",
    selected_validation["best_f1_threshold"],
)
display(selected_validation["deciles"])


Selected from validation: baseline_reproduction
Train ROC-AUC: 0.5082127283159514
Validation ROC-AUC: 0.4948672916974906
Best-F1 diagnostic threshold: 0.9985764032257011


,rows,conversions,observed_conversion_rate,avg_propensity,lift_vs_random
decile,,,,,
10,18211,145,0.007962,0.999829,1.067712
9,18210,125,0.006864,0.995148,0.920492
8,18210,117,0.006425,0.954236,0.861580
7,18211,148,0.008127,0.764083,1.089802
6,18210,142,0.007798,0.377743,1.045678
5,18210,130,0.007139,0.100629,0.957311
4,18211,132,0.007248,0.018442,0.971986
3,18210,128,0.007029,0.002545,0.942583
2,18210,151,0.008292,0.000203,1.111954


## 7. Final test evaluation

Do not keep choosing models after looking at test. If the result is poor, return to validation experiments and reserve a clean final holdout for the eventual final check.


In [ ]:
final_test = evaluate_model(
    selected_model,
    X_test,
    y_test,
    name=f"{selected_name} — TEST",
)

display(final_test["deciles"])


baseline_reproduction — TEST
  PR-AUC / Average Precision: 0.007156
  ROC-AUC:                    0.516547
  Base conversion rate:       0.006908


,rows,conversions,observed_conversion_rate,avg_propensity,lift_vs_random
decile,,,,,
10,18211,121,0.006644,0.999995,0.961818
9,18210,145,0.007963,0.999886,1.152655
8,18211,134,0.007358,0.998940,1.065154
7,18210,138,0.007578,0.993219,1.097009
6,18210,133,0.007304,0.963255,1.057263
5,18211,125,0.006864,0.832204,0.993613
4,18210,109,0.005986,0.475167,0.866478
3,18211,122,0.006699,0.120821,0.969767
2,18210,113,0.006205,0.011993,0.898276


## 8. Persist the final selected candidate without overwriting experiment artifacts

The final candidate is stored under this run ID. Earlier runs and the four per-experiment artifacts remain untouched.


In [ ]:
selected_config = next(
    config for config in experiments if config.name == selected_name
)

final_dir = RUN_DIR / "selected_final"
final_dir.mkdir(parents=True, exist_ok=True)

metadata = {
    "run_id": RUN_ID,
    "target": TARGET,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "model_features": MODEL_FEATURES,
    "train_cutoff": str(split.train_cutoff),
    "validation_cutoff": str(split.validation_cutoff),
    "selected_experiment": selected_config.as_dict(),
    "train_metrics": {
        "pr_auc": selected_train["pr_auc"],
        "roc_auc": selected_train["roc_auc"],
        "base_conversion_rate": selected_train["base_conversion_rate"],
    },
    "validation_metrics": {
        "pr_auc": selected_validation["pr_auc"],
        "roc_auc": selected_validation["roc_auc"],
        "best_f1": selected_validation["best_f1"],
        "best_f1_threshold": selected_validation["best_f1_threshold"],
        "precision_at_best_f1": selected_validation["precision_at_best_f1"],
        "recall_at_best_f1": selected_validation["recall_at_best_f1"],
    },
    "test_metrics": {
        "pr_auc": final_test["pr_auc"],
        "roc_auc": final_test["roc_auc"],
        "base_conversion_rate": final_test["base_conversion_rate"],
    },
}

model_path, metadata_path = save_model_and_metadata(
    selected_model,
    final_dir,
    metadata,
    model_filename="model.joblib",
    metadata_filename="metadata.json",
)

print("Final model saved to:", model_path)
print("Final metadata saved to:", metadata_path)


Final model saved to: /content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/experiments/20261004_210515/selected_final/model.joblib
Final metadata saved to: /content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/experiments/20261004_210515/selected_final/metadata.json


## 9. Produce scored test rows for the future Next Best Action layer


In [ ]:
scored_test = build_scored_test(
    split.test_df,
    final_test["probabilities"],
    TARGET,
)

scored_path = save_scored_test(
    scored_test,
    final_dir,
    filename="scored_test.parquet",
)

print("Scored rows:", f"{len(scored_test):,}")
print("Saved to:", scored_path)
print("Full run artifacts:", RUN_DIR)
display(scored_test.head())


Scored rows: 182,105
Saved to: /content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/experiments/20261004_210515/selected_final/scored_test.parquet
Full run artifacts: /content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/experiments/20261004_210515


,customer_id,campaign_id,send_date,country,promoted_product,send_channel,had_conversion,send_cost,conversion_value,predicted_conversion_probability
1031925,CLI-GNKSEFE9FWEN,CMP-LIEBCB6I4IWC,2026-01-15 19:13:29,México,Tarjeta Crédito,Push,0,0.0002,0.0,0.999813
1031926,CLI-9F8999U5NNLP,CMP-173ISW5CHVBB,2026-01-15 19:13:59,México,Préstamo Personal,Email,0,0.0039,0.0,0.077361
1031927,CLI-486AFWFW38JL,CMP-173ISW5CHVBB,2026-01-15 19:17:28,México,Préstamo Personal,Email,0,0.0039,0.0,0.946937
1031928,CLI-OZ7SV7QCUPVP,CMP-LIEBCB6I4IWC,2026-01-15 19:18:46,Argentina,Tarjeta Crédito,Push,0,0.0000,0.0,0.997929
1031929,CLI-97JLTJS1SPNQ,CMP-OZ3ECV86GFA8,2026-01-15 19:19:10,México,Tarjeta Crédito,Email,0,0.0000,0.0,0.999814


## Next step

Only after the propensity ranking is useful enough should notebook 06 / Next Best Action:

1. generate eligible Product × Channel candidate actions;
2. score candidates with the selected propensity model;
3. combine propensity, send cost, expected conversion value and business rules;
4. select or explain the recommended action.
